In [1]:
# 数据处理库
import sys
import json
import logging
import pandas as pd
import numpy as np
from dataset_deduplicate_utils import*
from datetime import datetime

In [2]:
class Config:
    
    TIMESTAMP = datetime.now().strftime('%y%m%d') 

    IN_DIR = '../data/input'
    IN_OUT_DIR = '../data/in_out'
    OUT_DIR = '../data/output'
    TEMP_DIR = '../tmp_data'

    config_path = '../configs/pretreatment_config.json'

    try:
        with open(config_path, 'r', encoding='utf-8') as f:
            _raw_config = json.load(f)
            _raw_dataset_name = _raw_config.get("RAW_DATASET_NAME")
            _property = _raw_config.get("PROPERTY")
            _state_var = _raw_config.get("STATE_VAR", "").strip()
            _reference = _raw_config.get("REFERENCE")
            
    except FileNotFoundError:
        logging.warning(f"配置文件未找到: {config_path}")    


    _base_name = os.path.splitext(_raw_dataset_name)[0]

    MISSING_DATA_MAG_DIR = os.path.join(TEMP_DIR, _base_name, 'mis_data_mag')
    DEDUPLICATE_DIR = os.path.join(TEMP_DIR, _base_name, 'deduplicate')

    # 输入文件路径
    DATADET_WITH_IDS = os.path.join(IN_OUT_DIR, f'{_base_name}_{TIMESTAMP}_raw-ds_add-ids_dd.xlsx')

    # 输出文件路径
    DATASET_DEDUPLICATED = os.path.join(IN_OUT_DIR, f'{_base_name}_{TIMESTAMP}_ds-with-ids_dd_fg.xlsx')
    DATASET_DEDUPLICATED_OUT = os.path.join(OUT_DIR, f'{_base_name}_{TIMESTAMP}_out_ds-dd.xlsx')

In [3]:
TIMESTAMP = datetime.now().strftime('%y%m%d') 

# 【解耦修改】从 Config 类动态读取配置变量
property_col = Config._property
ref_col = Config._reference
state_var_col = Config._state_var

# 【解耦修改】动态构建 essential_cols
essential_cols = ['c1', 'c2', 'x1', 'x2']
if property_col: essential_cols.append(property_col)
if ref_col: essential_cols.append(ref_col)
if state_var_col: essential_cols.append(state_var_col)

# 【配置】 定义组分名称列 (使用 SMILES 作为唯一标识)
# 注意：这三列将用于生成指纹，并在排序中起决定性作用
str_cols = [
    'c1_ismiles',
    'c2_ismiles',
    'c3_ismiles'
]

# 【配置】 定义组分比例列
# 注意：这里只包含比例 (x)，不能包含温度 (t)
num_cols = [
    'x1',
    'x2', 
    'x3'
]

# 中间结果输出目录
interim_output_dir = Config.DEDUPLICATE_DIR

# 定义处理完后要删除的辅助列 (例如原始的 c1, c2 名称列，如果不再需要)
drop_columns = ['c1', 'c2', 'c3']

# =============================================================================
# 2. 主程序执行流程
# =============================================================================

if __name__ == "__main__":
    print("="*50)
    print(f"开始执行数据去重流程")
    print(f"工作目录: {os.getcwd()}")
    print("="*50)

    # 1. 读取文件

    original_df = pd.read_excel(Config.DATADET_WITH_IDS)
    # 2. 调用 utils 中的去重函数
    # 注意参数的对应关系
    result_df = process_dataset_deduplication(
        original_df=original_df,
        essential_cols=essential_cols,
        str_cols=str_cols,       # 传入组分标识列
        num_cols=num_cols,       # 传入比例列 
        property_col=property_col, # 动态目标属性
        drop_columns=drop_columns,
        output_dir=interim_output_dir,
        ref_col=ref_col,         # 动态参考文献列名
        state_var_col=state_var_col   # 动态状态变量列 (如温度)
    )

    # 3. 生成质量报告
    print("\n正在生成数据质量报告...")
    qlty_sum_output_path = os.path.join(Config.MISSING_DATA_MAG_DIR, f'{TIMESTAMP}_ds-dd_qlty-sum.xlsx')
    
    try:
        data_quality_summary(result_df, qlty_sum_output_path)
    except Exception as e:
        print(f"生成质量报告时发生错误: {e}")

    # 4. 保存最终结果
    print("\n正在保存最终结果...")
    result_df.to_csv(Config.DATASET_DEDUPLICATED, index=False)
    result_df.to_csv(Config.DATASET_DEDUPLICATED_OUT, index=False)

    print("-" * 30)
    print("全部流程执行完毕。")
    print(f"最终结果文件: {Config.DATASET_DEDUPLICATED_OUT}")
    print("-" * 30)

开始执行数据去重流程
工作目录: d:\envs\codes\DES-DataAlchemy\dataset_pretreatment\src
=== [Utils] 1. 缺失值处理 ===
未检测到关键列的缺失值。
剩余行数: 5790

=== [Utils] 2. 生成规范化 Key (解决 A+B vs B+A 问题，包含温度: t) ===
正在生成唯一标识键...

=== [Utils] 3. 识别重复数据 ===
涉及重复的总行数: 732
正在优化 Excel 格式 (合并重复组编号)...
重复数据详情已保存至: ../tmp_data\02_vis_raw-dataset\deduplicate\raw-ds_dd_details.xlsx (已合并单元格)

=== [Utils] 4. 合并重复数据 ===
已将 732 行重复数据压缩为 365 条唯一记录。

=== [Utils] 5. 生成最终数据集 ===
最终数据集构建完成。总行数: 5423

正在生成数据质量报告...

正在保存最终结果...
------------------------------
全部流程执行完毕。
最终结果文件: ../data/output\02_vis_raw-dataset_260316_out_ds-dd.xlsx
------------------------------
